# surface u and v evolution

momentum side of the sim response. T gets most of the attention but the forcing is τ too so here's what the sim does with it.

In [ ]:
using JLD2
using Plots

gr()
default(size=(900, 500), linewidth=2, grid=true)

const ROOT = normpath(joinpath(@__DIR__, ".."))
const SITES = ["lat30lon-50", "lat-25lon-10", "lat-45lon80", "lat0lon-140", "lat30lon-150", "lat40lon-25"]

function paths(site)
    v1_qt = joinpath(ROOT, "output/era5/ml_forced_30day_glorysinit_qt_$(site).jld2")
    if site == "lat30lon-50"
        isfile(v1_qt) || (v1_qt = joinpath(ROOT, "output/era5/ml_forced_30day_glorysinit_qt.jld2"))
    end
    (v1_qt=v1_qt,
     v2_qt=joinpath(ROOT, "output/era5/ml_forced_30day_v2_qt_$(site).jld2"),
     glorys=site == "lat30lon-50" ?
        joinpath(ROOT, "data/generated/glorys_processed.jld2") :
        joinpath(ROOT, "data/generated/glorys_processed_$(site).jld2"))
end

load_if(p) = isfile(p) ? JLD2.load(p) : nothing

## surface u all sites

In [ ]:
function tiny_u_plot(site)
    p = paths(site)
    plt = plot(title=site, xlabel="Day", ylabel="Surface u (m/s)", titlefontsize=10)
    for (src, lbl, color) in [(load_if(p.v1_qt),"v1",:orange),(load_if(p.v2_qt),"v2",:red)]
        src === nothing && continue
        t = src["saved_times"] ./ 86400
        y = [pr[end] for pr in src["u_profiles"]]
        plot!(plt, t, y, label=lbl, color=color, lw=1.5)
    end
    g = load_if(p.glorys)
    if g !== nothing
        y = [pr[1] for pr in g["u_profiles"][1:min(31, length(g["u_profiles"]))]]
        plot!(plt, 0:length(y)-1, y, label="GLORYS", color=:green, linestyle=:dash, lw=1.5)
    end
    plt
end

plot([tiny_u_plot(s) for s in SITES]..., layout=(3,2), size=(1200,900))

## surface v all sites

In [ ]:
function tiny_v_plot(site)
    p = paths(site)
    plt = plot(title=site, xlabel="Day", ylabel="Surface v (m/s)", titlefontsize=10)
    for (src, lbl, color) in [(load_if(p.v1_qt),"v1",:orange),(load_if(p.v2_qt),"v2",:red)]
        src === nothing && continue
        t = src["saved_times"] ./ 86400
        y = [pr[end] for pr in src["v_profiles"]]
        plot!(plt, t, y, label=lbl, color=color, lw=1.5)
    end
    g = load_if(p.glorys)
    if g !== nothing
        y = [pr[1] for pr in g["v_profiles"][1:min(31, length(g["v_profiles"]))]]
        plot!(plt, 0:length(y)-1, y, label="GLORYS", color=:green, linestyle=:dash, lw=1.5)
    end
    plt
end

plot([tiny_v_plot(s) for s in SITES]..., layout=(3,2), size=(1200,900))

## day 30 momentum table

In [ ]:
using Printf
println(rpad("site",14), "| u: v1   v2   GLY   | v: v1   v2   GLY")
println("-"^65)
for site in SITES
    p = paths(site)
    v1 = load_if(p.v1_qt); v2 = load_if(p.v2_qt); g = load_if(p.glorys)
    surf(src, k) = src === nothing ? NaN : src[k][end][end]
    gsurf(k) = g === nothing ? NaN : g[k][30][1]
    @printf("%-14s| %+6.3f %+6.3f %+6.3f | %+6.3f %+6.3f %+6.3f\n", site,
            surf(v1,"u_profiles"), surf(v2,"u_profiles"), gsurf("u_profiles"),
            surf(v1,"v_profiles"), surf(v2,"v_profiles"), gsurf("v_profiles"))
end